# Infotheory parity benchmark: NumPy port vs JIDT

Tests every estimator/measure in `pyspi/statistics/infotheory.py` against the JIDT class it replaced.

**Test signal**: bivariate AR(1), unidirectional coupling x->y, 10 seeds per cell.

**T**: {200, 800, 1600}.

**JIDT setup**: matches the historical pyspi `_setup()` (`BIAS_CORRECTION=false`, `NOISE_SEED=42`), JIDT defaults otherwise.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image
pd.set_option('display.float_format', lambda v: f'{v:+.3e}')
df = pd.read_csv('parity_results.csv')
summary = pd.read_csv('parity_summary.csv')
print(f'{len(df)} cells (estimator x measure x T x seed)')


## Summary table


In [ ]:
summary.sort_values(['measure', 'estimator', 'T'])


## Error vs T


In [ ]:
Image('parity_plot.png')


## Findings

### Machine-precision parity (rel_err ~1e-14)
- `gaussian/MI`, `gaussian/TE`, `kernel/MI`, `kernel/entropy`, `symbolic/TE` agree with JIDT to floating-point precision.

### Tiny systematic bias (5e-9 nats, expected)
- `gaussian/entropy`: constant offset of `0.5 * log(1 + 1e-8) = 5e-9` nats from the NumPy port's ridge regularisation `Sigma + 1e-8 * mean(diag) * I`. Deterministic analogue of JIDT's stochastic `NOISE_LEVEL_TO_ADD=1e-8`, documented in `_gaussian_log_det`.

### Convergent finite-sample agreement
- `kraskov/MI`, `kraskov/TE`: KSG-family. Absolute error ~2e-3 nats at T=1600, decreasing with T. Both implementations are unbiased KSG estimators; the residual gap is driven by (i) JIDT's tiny additive observation noise vs NumPy's `eps * (1 - 1e-10)` strict-inequality trick, (ii) `count - 1` vs `count` self-exclusion semantics. Convergence is the expected `O(1/sqrt(N))`.
- `kozachenko/entropy`: <1e-4 nats at T=1600.
- `kernel/TE`: 5e-5 nats at T=1600.

### Bug found + fixed during this benchmark
Initial run showed `kernel/entropy` with a constant ~0.21 nats gap independent of T. Root cause: `KernelEntropyCalculator` standardised the data when `NORMALISE=true` but forgot to add the scale-correction term `d * log2(prod(std))` to the entropy. JIDT instead leaves the data raw and rescales the bandwidth (`kernelWidthsInUse = w * std`), which gives the full entropy with the std term built into the volume `(2*w*std)^d`. The two approaches are equivalent only with that correction.

Fix applied to `pyspi/statistics/infotheory.py::KernelEntropyCalculator.computeAverageLocalOfObservations` — add `+ sum_d log2(std_d)` when normalise=true. Gap drops from +0.21 to 0.00 +/- 0.00 across all T. MI/TE were unaffected because the std factor cancels in the count ratios.

Reference: Kantz & Schreiber, *Nonlinear Time Series Analysis* (1997); Schreiber (2000); Lizier 2014 JIDT paper.

## Convergence expectation

At T=1600, the typical AR(1) MI/TE we're estimating is ~0.03-0.10 nats with KSG estimator standard error of order `1/sqrt(k * N) ~ 0.012`. The implementation-vs-implementation absolute gap we measure (~2e-3 nats) is one order of magnitude below the inherent estimator noise -- the two implementations agree to within ~10% of one estimator standard deviation. T=200 already resolves the patterns; longer T not needed for this question.
